# Agent Debug Notebook

Инструмент для интерактивной отладки агента. Запускай ячейки по порядку.

**Быстрый старт:**
1. Запусти «Imports & setup» (Cell 1)
2. Напиши вопрос в Cell 3 и запусти
3. Анализируй трейс в следующих ячейках

In [1]:
# Cell 1 — импорты и логирование
import json
import logging
import os
import sys

# добавляем корень проекта в PYTHONPATH
project_root = (
    os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# минимальное логирование: только WARNING от сторонних либ, INFO от нашего кода
logging.basicConfig(level=logging.WARNING, format="%(name)s | %(message)s")
logging.getLogger("agent").setLevel(logging.INFO)
logging.getLogger("retriever").setLevel(logging.INFO)

from agent.graph import ask_debug

print("✓ Готово. Импорты загружены.")

/Users/mikhail/Projects/RAGv2/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✓ Готово. Импорты загружены.


In [2]:
# Cell 3 — задать вопрос агенту и получить трейс
QUESTION = "Сколько детей у Галаевой?"  # ← меняй здесь

print(f"Вопрос: {QUESTION}")
print("Обрабатываю...")

response, trace = ask_debug(QUESTION)

print(f"\nГотово! Итераций: {response.iterations}, chunks: {response.chunks_used}")

Вопрос: Сколько детей у Галаевой?
Обрабатываю...


huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Готово! Итераций: 1, chunks: 15


In [4]:
# Cell 4 — полный трейс (сводный вывод)
trace.display()

╔══════════════════════════════════════════════════════════════════╗
║  AGENT TRACE                                                     ║
╠══════════════════════════════════════════════════════════════════╣
║  Вопрос:    Сколько детей у Галаевой?                             ║
║  Thread:    e3266576-86fb-434e-876a-16906542561d                  ║
║  Итог:      13390ms | iter=1 | chunks=15                          ║
╚══════════════════════════════════════════════════════════════════╝

── LLM [agent] (2141ms) ──────────────────────────────────────────
  Сообщений на входе: 2
    [system] Ты — персональный помощник по базе знаний Obsidian.↵↵# Главное правило↵Отвечай ТОЛЬКО на основе информации из базы знани
    [human] Сколько детей у Галаевой?
  Tool calls (1):
    → search_knowledge_base({"query": "Галаева дети", "bm25_terms": "Галаева"})

── TOOL #1: search_knowledge_base (7173ms) ───────────────────────
  query: Галаева дети
  bm25_terms: Галаева
  Найдено: 15 документов
    [1] А+ и Ре

In [ ]:
# Cell 5 — drill-down: retrieval (что нашли и с каким скором)
search_calls = [t for t in trace.tool_calls if t.name == "search_knowledge_base"]

if not search_calls:
    print("Поиск не вызывался.")
else:
    for tool_ev in search_calls:
        print(f"\n{'=' * 60}")
        print(f"Tool call #{tool_ev.order}  |  {tool_ev.latency_ms:.0f}ms")
        print(f"  query:      {tool_ev.args.get('query', '')}")
        print(f"  bm25_terms: {tool_ev.args.get('bm25_terms')}")

        if tool_ev.retrieved_docs:
            print(f"\n  Найдено {len(tool_ev.retrieved_docs)} документов:")
            for doc in tool_ev.retrieved_docs:
                sec = f" > {doc.section}" if doc.section else ""
                bar = "█" * int(doc.score * 20)  # мини-баргraph
                print(f"  [{doc.index:2d}] score={doc.score:.4f} {bar:<20} {doc.source}{sec}")
        else:
            print(f"  Результат: {tool_ev.result[:200]}")

In [ ]:
# Cell 6 — drill-down: LLM-вызовы (контекст, который видел LLM)
for i, llm_ev in enumerate(trace.llm_calls, 1):
    print(f"\n{'=' * 60}")
    print(f"LLM call #{i}  |  node={llm_ev.node}  |  {llm_ev.latency_ms:.0f}ms")
    print(f"  Сообщений на входе: {len(llm_ev.messages_in)}")

    # показываем все входные сообщения
    for msg in llm_ev.messages_in:
        role = msg["role"]
        content = msg["content"]
        # system-промпт обрезаем
        if role == "system" and len(content) > 200:
            content = content[:200] + "... [system prompt обрезан]"
        # tool-результаты обрезаем
        if role == "tool" and len(content) > 400:
            content = content[:400] + "... [обрезано]"
        print(f"\n  [{role.upper()}]")
        print("  " + content.replace("\n", "\n  "))

    # ответ LLM
    if llm_ev.tool_calls:
        print(f"\n  → Вызывает инструменты: {[tc['name'] for tc in llm_ev.tool_calls]}")
        for tc in llm_ev.tool_calls:
            print(f"    {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)})")
    if llm_ev.response:
        print("\n  ← Ответ LLM:")
        print("  " + llm_ev.response[:500].replace("\n", "\n  "))

In [ ]:
# Cell 7 — latency breakdown (где тормозит)
total_llm_ms = sum(e.latency_ms for e in trace.llm_calls)
total_tool_ms = sum(e.latency_ms for e in trace.tool_calls)
other_ms = trace.total_latency_ms - total_llm_ms - total_tool_ms

print(f"Total latency: {trace.total_latency_ms:.0f}ms")
print(f"  LLM calls:   {total_llm_ms:.0f}ms  ({total_llm_ms / trace.total_latency_ms * 100:.1f}%)")
for e in trace.llm_calls:
    print(f"    [{e.node}] {e.latency_ms:.0f}ms")
print(
    f"  Tool calls:  {total_tool_ms:.0f}ms  ({total_tool_ms / trace.total_latency_ms * 100:.1f}%)"
)
for e in trace.tool_calls:
    print(f"    #{e.order} {e.name} {e.latency_ms:.0f}ms")
print(f"  Overhead:    {other_ms:.0f}ms  (граф, checkpointing, etc.)")

# мини-гистограмма
print()

print("Визуализация:")
scale = 40 / trace.total_latency_ms
print(f"  LLM  {'█' * int(total_llm_ms * scale)}  {total_llm_ms:.0f}ms")
print(f"  Tool {'█' * int(total_tool_ms * scale)}  {total_tool_ms:.0f}ms")
print(f"  Misc {'█' * max(1, int(other_ms * scale))}  {max(0, other_ms):.0f}ms")

In [ ]:
# Cell 8 — сохранить трейс в JSON для дальнейшего анализа
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"trace_{timestamp}.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(trace.to_dict(), f, ensure_ascii=False, indent=2)

print(f"Трейс сохранён: {filename}")